# 05 · B3: per què les reparacions de bateria acaben sense arreglar

1. **Resultat principal (MobiFix):** com acaben les reparacions de bateria, comparades amb les de pantalla.
2. **Per què:** una precodificació automàtica proposa un motiu per a cada final de vida de bateria, i **una persona** la revisa a `dades/processades/revisio_bateria_fi_vida.csv` (columna `codi_revisat`). La pregunta és sobretot si el motiu **consta** o no, i això no depèn de si el tipus d'avaria és exacte, així que fem servir tots els finals de vida de bateria (també els del classificador).

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

import comu

rng = np.random.default_rng(42)

dades = comu.carrega_classificats()
estat = {"Fixed": "Reparat", "Repairable": "Reparable", "End of life": "Final de vida"}
dades["resultat"] = dades["repair_status"].map(estat)
ORDRE_RES = ["Reparat", "Reparable", "Final de vida"]

def resultats(df):
    t = pd.crosstab(df["grup_avaria"], df["resultat"]).loc[["Bateria i alimentació", "Pantalla"], ORDRE_RES]
    return t.join((t.div(t.sum(axis=1), axis=0) * 100).round(1).add_suffix(" %"))

mf = dades[dades["font_avaria"] == "MobiFix"]
print("Resultat principal (MobiFix):")
display(resultats(mf))
print("Complement no validat (conjunt classificat):")
resultats(dades)

Resultat principal (MobiFix):


resultat,Reparat,Reparable,Final de vida,Reparat %,Reparable %,Final de vida %
grup_avaria,,,,,,
Bateria i alimentació,78,96,26,39.0,48.0,13.0
Pantalla,323,232,29,55.3,39.7,5.0


Complement no validat (conjunt classificat):


resultat,Reparat,Reparable,Final de vida,Reparat %,Reparable %,Final de vida %
grup_avaria,,,,,,
Bateria i alimentació,210,203,89,41.8,40.4,17.7
Pantalla,514,364,87,53.3,37.7,9.0


## Figura 3

La figura la genera `scripts/07_b3_resultat.py` (Altair), amb el disseny de `disseny/b3_resultat/`: com acaben les reparacions de bateria i de pantalla a MobiFix. Els motius dels finals de vida es resumeixen més avall i van al text de l'entrada.

In [2]:
import importlib.util

spec = importlib.util.spec_from_file_location("b3", "../scripts/07_b3_resultat.py")
b3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(b3)

taula_b3 = b3.derive(b3.load(b3.DATA_PATH))
display(taula_b3[["component", "resultat", "n", "total", "pct"]].round(1))
grafic = b3.build_chart(taula_b3)
b3.save(grafic, b3.OUT_DIR, b3.SLUG)
grafic

loaded 5652 rows from /home/sigkaio/Desktop/clase/PEF1/vincle-obsolescencia/posts/vida-material/_analisi/scripts/../dades/processades/mobils_classificats.csv


,component,resultat,n,total,pct
0,Bateria,Final de vida,26,200,13.0
1,Bateria,Pendent,96,200,48.0
2,Bateria,Reparat,78,200,39.0
3,Pantalla,Final de vida,29,584,5.0
4,Pantalla,Pendent,232,584,39.7
5,Pantalla,Reparat,323,584,55.3


alt.VConcatChart(...)

## Precodificació dels finals de vida

Llibre de codis (s'aplica el primer que coincideix, en aquest ordre):

| Codi | Què vol dir |
|---|---|
| `inflada` | Bateria inflada, vessada o problema de seguretat |
| `no_era_bateria` | L'avaria era la placa, el port o un altre component |
| `no_sobre` | No es pot obrir, la bateria va enganxada o no es pot canviar, o calen eines especials |
| `peca` | No hi ha la peça, s'ha de demanar o és massa cara |
| `a_casa` | La persona compra la peça i la canvia ella mateixa |
| `no_val_pena` | El mòbil és massa vell o no surt a compte (cost i benefici) |
| `no_consta` | El text no diu per què |

In [3]:
import re

CODIS = {
    "inflada": comu.SIMPTOMES["inflada"] + r"|ausgelaufen|leaked|lekt|lekkage",
    "no_era_bateria": r"motherboard|mainboard|logic ?board|\bboard\b|platine|moederbord|carte mère|placa base|\bport\b|buchse|charging ic"
                      r"|ruhestrom|gleiches verhalten|same (?:problem|behaviou?r) with",
    "no_sobre": r"glued|geklebt|verklebt|lijm|can.?t open|cannot open|could not open|nicht öffnen|niet open|ouvrir|no se puede abrir|åbne"
                r"|inchangeable|non.?remov|not removable|nicht wechselbar|verbaut|special(?:ist)? tools|spezialwerkzeug",
    "peca": r"spare|\bparts?\b|ersatz|onderdeel|pièce|repuesto|reservedel|bestell|order|lieferbar|leverbaar|expensive|teuer|duur|cher|commander|commande",
    "a_casa": r"at home|zu hause|zuhause|thuis|à la maison|en casa|selbst|zelf|herself|himself|themselves|will replace",
    "no_val_pena": r"not worth|lohnt|loont niet|rentable|no merece|too old|zu alt|te oud|trop vieux|unwirtschaftlich"
                   r"|kosten|baten|\bcosts?\b|wirtschaftlich|economic",
}

def codifica(text):
    t = str(text).lower()
    trobats = [c for c, patro in CODIS.items() if re.search(patro, t)]
    return (trobats[0] if trobats else "no_consta"), ";".join(trobats)

fi_vida = dades[(dades["grup_avaria"] == "Bateria i alimentació") & (dades["repair_status"] == "End of life")
                & dades["problem"].notna()].copy()
fi_vida[["codi_auto", "codis_auto_tots"]] = fi_vida["problem"].apply(lambda t: pd.Series(codifica(t)))
print(f"Finals de vida de bateria: {len(fi_vida)} ({(fi_vida['font_avaria'] == 'MobiFix').sum()} de MobiFix)")
fi_vida["codi_auto"].value_counts()

Finals de vida de bateria: 89 (26 de MobiFix)


codi_auto
no_consta         67
no_era_bateria     7
inflada            6
no_val_pena        4
peca               3
no_sobre           2
Name: count, dtype: int64

In [4]:
fitxer = comu.PROCESSADES / "revisio_bateria_fi_vida.csv"
if not fitxer.exists():
    (fi_vida[["id", "font_avaria", "problem", "repair_barrier_if_end_of_life", "codi_auto", "codis_auto_tots"]]
     .assign(codi_revisat="")
     .to_csv(fitxer, index=False))
    print(f"Generat {fitxer.name}: una persona ha d'omplir codi_revisat amb un dels codis")
print("Codis possibles:", list(CODIS) + ["no_consta"])

Codis possibles: ['inflada', 'no_era_bateria', 'no_sobre', 'peca', 'a_casa', 'no_val_pena', 'no_consta']


## Resum: consta el motiu?

In [5]:
revisio = pd.read_csv(fitxer)
fet = revisio["codi_revisat"].notna().sum()
if fet < len(revisio):
    print(f"Revisió pendent ({fet} de {len(revisio)}): de moment es fa servir la precodificació automàtica.")
    codi = revisio["codi_auto"]
else:
    print(f"Revisió feta. La precodificació coincideix en el {(revisio['codi_auto'] == revisio['codi_revisat']).mean():.0%} dels casos.")
    codi = revisio["codi_revisat"]
taula = codi.value_counts().to_frame("n").assign(pct=lambda d: (d["n"] / d["n"].sum() * 100).round(0))
display(taula)
print(f"Amb motiu al camp dels voluntaris: {revisio['repair_barrier_if_end_of_life'].notna().sum()} de {len(revisio)}")
revisio["repair_barrier_if_end_of_life"].value_counts()

Revisió pendent (0 de 89): de moment es fa servir la precodificació automàtica.


,n,pct
codi_auto,,
no_consta,67,75.0
no_era_bateria,7,8.0
inflada,6,7.0
no_val_pena,4,4.0
peca,3,3.0
no_sobre,2,2.0


Amb motiu al camp dels voluntaris: 32 de 89


repair_barrier_if_end_of_life
Spare parts too expensive           9
No way to open product              6
Spare parts not available           6
Lack of equipment                   5
Repair information not available    4
Product too worn out                2
Name: count, dtype: int64